# 03 — Signal Analysis

Reads pre-processed data from `data/processed/` (built by `process_data.py`) and runs the event study:
- Do spike days predict abnormal next-day returns?
- Do they predict elevated next-day volume?
- Which sectors show the strongest signal?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load processed data

In [ ]:
signal_df = pd.read_csv('../data/processed/signal_table.csv', parse_dates=['date'])

clean = signal_df.dropna(subset=['next_day_abnormal_return', 'next_day_volume_ratio'])
print(f"Signal table: {len(clean):,} rows with full data")
print(f"Spike days: {clean['is_spike'].sum():,}")
print(f"Date range: {clean['date'].min().date()} to {clean['date'].max().date()}")
clean.head()

## 2. Core question: do spike days produce different next-day returns?

Two-sample t-test comparing mean next-day abnormal return on spike vs. non-spike days.

In [ ]:
spikes    = clean[clean['is_spike']]['next_day_abnormal_return']
no_spikes = clean[~clean['is_spike']]['next_day_abnormal_return']

t_stat, p_val = stats.ttest_ind(spikes, no_spikes, equal_var=False)

print("Next-day abnormal return — spike vs. non-spike days")
print(f"  Spike days    n={len(spikes):,}  mean={spikes.mean():.4f}  std={spikes.std():.4f}")
print(f"  Non-spike     n={len(no_spikes):,}  mean={no_spikes.mean():.4f}  std={no_spikes.std():.4f}")
print(f"  t-stat={t_stat:.3f}  p-value={p_val:.4f}")
print(f"  {'SIGNIFICANT at 5%' if p_val < 0.05 else 'Not significant at 5%'}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(no_spikes.clip(-0.1, 0.1), bins=80, alpha=0.6, label='Non-spike days', density=True)
ax.hist(spikes.clip(-0.1, 0.1), bins=40, alpha=0.7, label='Spike days', density=True)
ax.axvline(spikes.mean(), color='tab:orange', linestyle='--', label=f'Spike mean ({spikes.mean():.4f})')
ax.axvline(no_spikes.mean(), color='tab:blue', linestyle='--', label=f'Non-spike mean ({no_spikes.mean():.4f})')
ax.set_xlabel('Next-day abnormal return')
ax.set_ylabel('Density')
ax.set_title('Next-day abnormal return: spike vs. non-spike days')
ax.legend()
plt.tight_layout()
plt.savefig('../results/return_distribution_spike_vs_nonspike.png', dpi=150)
plt.show()

## 3. Volume signal: do spikes predict elevated next-day trading volume?

In [ ]:
vol_spikes    = clean[clean['is_spike']]['next_day_volume_ratio']
vol_no_spikes = clean[~clean['is_spike']]['next_day_volume_ratio']

t_vol, p_vol = stats.ttest_ind(vol_spikes, vol_no_spikes, equal_var=False)

print("Next-day volume ratio — spike vs. non-spike days")
print(f"  Spike days    n={len(vol_spikes):,}  mean={vol_spikes.mean():.4f}  median={vol_spikes.median():.4f}")
print(f"  Non-spike     n={len(vol_no_spikes):,}  mean={vol_no_spikes.mean():.4f}  median={vol_no_spikes.median():.4f}")
print(f"  t-stat={t_vol:.3f}  p-value={p_vol:.4f}")
print(f"  {'SIGNIFICANT at 5%' if p_vol < 0.05 else 'Not significant at 5%'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label, data in zip(axes, ['Non-spike days', 'Spike days'], [vol_no_spikes, vol_spikes]):
    ax.hist(data.clip(0, 5), bins=60, edgecolor='white')
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.3f}')
    ax.axvline(1.0, color='gray', linestyle=':', label='baseline')
    ax.set_xlabel('Next-day volume ratio')
    ax.set_title(label)
    ax.legend()
plt.suptitle('Next-day volume ratio: spike vs. non-spike days')
plt.tight_layout()
plt.savefig('../results/volume_ratio_spike_vs_nonspike.png', dpi=150)
plt.show()

## 4. Sector breakdown: where is the signal strongest?

In [ ]:
def sector_ttest(df, outcome_col):
    rows = []
    for sector, grp in df.groupby('sector'):
        s = grp[grp['is_spike']][outcome_col].dropna()
        ns = grp[~grp['is_spike']][outcome_col].dropna()
        if len(s) < 10 or len(ns) < 10:
            continue
        t, p = stats.ttest_ind(s, ns, equal_var=False)
        rows.append({
            'sector': sector,
            'spike_mean': s.mean(),
            'nonspike_mean': ns.mean(),
            'diff': s.mean() - ns.mean(),
            'n_spike': len(s),
            'p_value': p,
        })
    return pd.DataFrame(rows).sort_values('p_value')

return_by_sector = sector_ttest(clean, 'next_day_abnormal_return')
print("Abnormal return signal by sector:")
print(return_by_sector.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tab:green' if d > 0 else 'tab:red' for d in return_by_sector['diff']]
bars = ax.barh(return_by_sector['sector'], return_by_sector['diff'] * 100, color=colors)
ax.bar_label(bars, fmt='%.3f%%', padding=3)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Mean difference in next-day abnormal return (spike - non-spike), %')
ax.set_title('Wikipedia spike signal by sector')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/signal_by_sector.png', dpi=150)
plt.show()

## 5. Top spike events (highest views) and their next-day outcomes

In [ ]:
top_spikes = (
    clean[clean['is_spike']]
    .sort_values('views', ascending=False)
    .head(30)[['date', 'ticker', 'sector', 'views', 'next_day_abnormal_return', 'next_day_volume_ratio']]
)

top_spikes = top_spikes.copy()
top_spikes['next_day_abnormal_return'] = top_spikes['next_day_abnormal_return'].map('{:.2%}'.format)
top_spikes['next_day_volume_ratio'] = top_spikes['next_day_volume_ratio'].map('{:.2f}x'.format)
top_spikes['views'] = top_spikes['views'].map('{:,.0f}'.format)

print("Top 30 spike events by Wikipedia views:")
print(top_spikes.to_string(index=False))

## 6. Summary

In [ ]:
summary = (
    clean.groupby('is_spike')
    .agg(
        n=('next_day_abnormal_return', 'count'),
        mean_abnormal_return=('next_day_abnormal_return', 'mean'),
        median_abnormal_return=('next_day_abnormal_return', 'median'),
        mean_volume_ratio=('next_day_volume_ratio', 'mean'),
        median_volume_ratio=('next_day_volume_ratio', 'median'),
    )
)
summary.index = ['Non-spike', 'Spike']
summary